TABLEAU A CONSTRUIRE

* lignes : INDEX 1 = dataset/model ; INDEX 2 = $\kappa_f$ ; INDEX 3 = 'hyperparameter value': n=... puis imbalance=... puis $\delta$=... 

* colonnes = normalized Areas Between Curves (ABC, between bounds curve and test metric curve, normalized by number of common $\theta$ points on x-axis): R0/1, R0/1, FPP, FPP, ..., moyennes calculées sur 10 splits de CV



In [1]:
import os
import sys
current_dir = os.getcwd()
root_path = os.path.abspath(os.path.join(current_dir, '..'))
if root_path not in sys.path:
    sys.path.append(root_path)
import torch
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import torch.nn as nn
import numpy as np
import pickle
import pandas as pd
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import math
import scipy.special
import random as rd
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import VGG16_Weights
from tqdm import tqdm
import torch.optim.lr_scheduler as lr_scheduler
from python_scripts.sgr_utils import *
from python_scripts.plotting import *
from python_scripts.preprocessing import *
from scipy.special import gammaln
import warnings
warnings.filterwarnings("ignore")
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 150

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: False


In [2]:
datasets_paths = [
    'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/BC/sgr_set_log_reg',
    'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/CIFAR2/sgr_set_cnn',
    'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/CIFAR2/sgr_set_resnet',
    'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/CIFAR2/sgr_set_cnn_MCD',
    # put resnet_MCD path here
    'C:/Users/Emilien JEMELEN/Documents/SGR/experiments/WSI/sgr_set_cnn'
]

In [ ]:
dicos = []

#for path in dataset_paths:

path = datasets_paths[1]

datasets = ['BC', 'CIFAR2', 'WSI']
models = ["log_reg", "cnn", "resnet"]

mod = models[int(np.where([(m in path) for m in models])[0][0])]
ds = datasets[int(np.where([(d in path) for d in datasets])[0][0])]
if 'MCD' in path:
    kappa = 'MCD'
else:
    kappa = 'SR'

original_ds = pickle.load(open(path, 'rb'))

print('Sample size proportions...')
for prop in [1, 3/4, 1/2, 1/4]:
    print('prop = ', prop)
    for metric in ['standard', 'FP', 'FN', 'FPR', 'FNR']:
        print("metric = ", metric)
        values = []
        for seed in tqdm(range(10)):
            df = original_ds.sample(frac=prop, random_state=seed)
            values.append(ABC(df, metric=metric))
        dicos.append({'dataset': ds,
                        'model': mod,
                        'kappa': kappa,
                        'metric': metric,
                        'param.': 'N',
                        'value': prop,
                        'ABC': np.mean(values)})

print('Imbalance rates...')
for imbalance in [3/4, 1/2, 1/4]:
    print('imbalance = ', imbalance)
    for metric in ['standard', 'FP', 'FN', 'FPR', 'FNR']:
        print('metric = ', metric)
        values = []
        for seed in tqdm(range(10)):
            df = generate_imbalanced_datasets(original_ds, [imbalance], seed=seed)
            values.append(ABC(df, metric=metric))
        dicos.append({'dataset': ds,
                      'model': mod,
                      'kappa': kappa,
                      'metric': metric,
                      'param.': 'imbalance',
                      'value': imbalance,
                      'ABC': np.mean(values)})
        
print(r'$\delta$ values...')
for delta in [0.001, 0.01, 0.1]:
    print('delta = ', delta)
    for metric in ['standard', 'FP', 'FN', 'FPR', 'FNR']:
        print('metric = ', metric)
        values = []
        for seed in tqdm(range(10)):
            df = original_ds.sample(frac=1, random_state=seed)
            values.append(ABC(df, metric=metric, delta=delta))
        dicos.append({'dataset': ds,
                        'model': mod,
                        'kappa': kappa,
                        'metric': metric,
                        'param.': 'delta',
                        'value': delta,
                        'ABC': np.mean(values)})

Sample size proportions...
prop =  1
metric =  standard


100%|██████████| 10/10 [01:33<00:00,  9.31s/it]


metric =  FP


100%|██████████| 10/10 [01:23<00:00,  8.35s/it]


metric =  FN


...

100%|██████████| 10/10 [01:38<00:00,  9.82s/it]


metric =  FN


100%|██████████| 10/10 [00:15<00:00,  1.52s/it]


metric =  FPR


 70%|███████   | 7/10 [01:29<00:35, 11.72s/it]

In [ ]:
res = pd.DataFrame(dicos)
display(res)